# CV Experiments Notebook

In [ ]:
import os
from pathlib import Path
ROOT = Path(os.getcwd()).parents[0]

In [ ]:
import numpy as np
import pandas as pd
import json
import matplotlib.pyplot as plt

import cv2
from PIL import Image

import torch
import torch.nn.functional as F
from torch import optim, nn, utils, Tensor
from torch.utils.data import DataLoader, Dataset
from torch.utils.data.dataset import random_split

import torchvision
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.models.inception import InceptionOutputs
import torchvision.models as models

import torchmetrics

import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from lightning.pytorch.loggers import CSVLogger

##  Cargar y procesar todos los datos a utilizar

### Cargar 'instances', 'captions' y 'categories' del dataset de COCO, para guardarlas en diccionarios

In [ ]:
# Functions to read and process COCO data from .json files

def load_json(path):
    """
    Load a JSON file and return it as a Python dictionary.

    Parameters:
        path (str): Path to the JSON file.

    Returns:
        dict: Parsed JSON content.
    """
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f) # JSON to Dict

def build_id_to_filename(coco_json):
    """
    Build a dictionary mapping image IDs to their corresponding file names.
    
    Parameters:
        coco_json (dict): Entire COCO annotation JSON.

    Returns:
        dict: Mapping {image_id: file_name}
    """
    # Map id to its filename
    return {img["id"]: img["file_name"] for img in coco_json["images"]}

def group_by_filename(items, id_to_filename, key="image_id"):
    """
    Group a list of annotation items by their image filename.

    Parameters:
        items (list): List of annotation items (dicts).
        id_to_filename (dict): Mapping {image_id: file_name}.
        key (str): The key in each item that holds the image ID. 

    Returns:
        dict: Mapping {file_name: list_of_items}
    """
    grouped = {} # final dict

    for item in items:
        # Extract the image_id from the item
        img_id = item[key]

        # Convert numeric image_id to filename (string)
        filename = id_to_filename[img_id]

        # Create a list if key doesn't exist, then append the item
        grouped.setdefault(filename, []).append(item)

    return grouped

In [ ]:
# Function to parse 'instances' from COCO

def parse_instances(json_path):
    """
    Parse a COCO instances annotation file (used for object detection
    and instance segmentation) and return a dictionary mapping each image
    filename to a list of its object annotations.

    Parameters:
        json_path (str): Path to COCO instances JSON file.

    Returns:
        dict: {filename: [object_annotation_dict, ...]}
    """

    # Load the full COCO JSON (images + object instance annotations)
    data = load_json(json_path)

    # Create the mapping helper from id to filename
    id_to_filename = build_id_to_filename(data)

    # Extract only the fields needed for object.
    objs = [
        {
            "image_id": ann["image_id"], # image ID
            "segmentation": ann["segmentation"], # polygon or RLE mask
            "area": ann["area"], # pixel area of the object
            "iscrowd": ann["iscrowd"], # crowd/cluster indicator
            "bbox": ann["bbox"], # [x, y, width, height]
            "category_id": ann["category_id"], # COCO class ID
            "id": ann["id"] # annotation ID
        }
        for ann in data["annotations"]
    ]

    # Return the group annotations by filename.
    return group_by_filename(objs, id_to_filename)



# Function to parse 'captions' from COCO

def parse_captions(json_path):
    """
    Parse a COCO captions annotation file and return a dictionary
    mapping each image filename to its list of captions.

    Parameters:
        json_path (str): Path to COCO captions JSON file.

    Returns:
        dict: {filename: [caption1, caption2, ...]}
    """
    # Load the full COCO JSON
    data = load_json(json_path)

    # Build a helper mapping id to filename
    id_to_filename = build_id_to_filename(data)

    # Extract only the useful fields for captioning
    captions_only = [{"image_id": ann["image_id"], "caption": ann["caption"]} for ann in data["annotations"]]

    # Group captions by image filename
    grouped = group_by_filename(captions_only, id_to_filename)

    # Now convert the grouped structure so each image maps directly
    for filename in grouped:
        grouped[filename] = [x["caption"] for x in grouped[filename]]

    # Return the clean final structure
    return grouped

In [ ]:
# Functions to parse and process 'categories' from COCO

def load_categories(json_path):
    """
    Load COCO categories from an annotation JSON file.

    Returns:
        list[dict[str]]
    """
    # Load the full COCO JSON
    data = load_json(json_path)

    # Extract only the fields needed for categories
    categories = [
        {
            "supercategory": cat["supercategory"],
            "id": cat["id"],
            "name": cat["name"],
        }
        for cat in data["categories"]
    ]

    # Return a list of dictionaries with the supercategory, id and name
    return categories

def build_super_to_categories(categories):
    """
    Group categories by their supercategory.

    Parameters:
        categories (list[dict]): Output of load_categories().

    Returns:
        dict: {supercategory_name: [{"id": <category_id>, "name": <category_name>}]}
    """
    super_to_categories = {}

    for cat in categories:
        # Get keys
        supercat = cat["supercategory"]
        cat_id = cat["id"]
        name = cat["name"]

        # Create entry dict to append on each supercategory
        entry = {"id": cat_id, "name": name}

        # Create a list if key doesn't exist, then append the item
        super_to_categories.setdefault(supercat, []).append(entry)

    return super_to_categories

def parse_categories(json_path):
    # Load raw category info
    categories = load_categories(json_path)

    # Get the dict
    super_to_categories = build_super_to_categories(categories)

    return super_to_categories

### Guardar datos en diccionarios

In [ ]:
# Routes to .json files (For instances and captions)

path_instances_train = f"{ROOT}\\raw_data\\annotations\\instances_train2017.json"
path_captions_train = f"{ROOT}\\raw_data\\annotations\\captions_train2017.json"

path_instances_val = f"{ROOT}\\raw_data\\annotations\\instances_val2017.json"
path_captions_val = f"{ROOT}\\raw_data\\annotations\\captions_val2017.json"

# path_info_test = f"{ROOT}\\raw_data\\annotations\\tests\\image_info_test-dev2017.json"
# path_info_test = f"{ROOT}\\raw_data\\annotations\\tests\\image_info_test2017.json"

In [ ]:
# Parse the .json files

insts_dict_train = parse_instances(path_instances_train)
capts_dict_train = parse_captions(path_captions_train)
categ_dict_train = parse_categories(path_instances_train)

# insts_dict_val = parse_instances(path_instances_val)
# capts_dict_val = parse_captions(path_captions_val)
# categ_dict_val = parse_categories(path_instances_val)

# insts_dict_test = parse_instances(path_info_test)
# capts_dict_test = parse_captions(path_info_test)
# categ_dict_test = parse_categories(path_info_test)

### Mostrar contenido dentro de los diccionarios

In [ ]:
# Function to print all the contents of the 'instances' dictionary

def display_annotations(instances_raw, limit_count=5):
    count = 0

    print(f"Directory Length: {len(instances_raw)}")

    for key, values in instances_raw.items():
        if count != limit_count:
            count += 1

            print(f"======== Image filename: {key} ========")
            
            for ann in instances_raw[key]:
                print("\n-------- Annotation --------")

                for field, value in ann.items():
                    print(f"{field}: {value}")
                        
            print(f"\n\n\n")
        
        else:
            break

In [ ]:
display_annotations(insts_dict_train)

In [ ]:
# Function to print all the contents of the 'captions' dictionary

def display_captions(captions_raw, limit_count=5):
    count = 0

    print(f"Directory Length: {len(captions_raw)}")

    for key, value in captions_raw.items():
        if count != limit_count:
            count += 1

            print(f"======== Image filename: {key} ========")
            
            for caption in captions_raw[key]:
                print(caption)
                        
            print("\n")
        
        else:
            break

In [ ]:
display_captions(capts_dict_train)

In [ ]:
# Function to print all the contents of the 'categories' dictionary

def display_categories(categories_raw):

    for key, values in categories_raw.items():
        print(f"======== SuperCategory: {key} ========")
        for cat in categories_raw[key]:
            print("\n-------- Category --------")

            for field, value in cat.items():
                print(f"{field}: {value}")

In [ ]:
display_categories(categ_dict_train)

### Obtener lista de nombres de imágenes del diccionario

In [ ]:
# Function to get a sorted list of keys from a dictionary

def get_keys_list(directory):
    keys_list = list(directory.keys())
    keys_list.sort()

    # print(len(keys_list))

    # for i in range(len(keys_list)):
      #   print(keys_list[i])
    
    return keys_list

In [ ]:
# Getting the lists of key from the dictionaries

insts_list_train = get_keys_list(insts_dict_train)
capts_list_train = get_keys_list(capts_dict_train)

# insts_list_val = get_keys_list(insts_dict_val)
# capts_list_val = get_keys_list(capts_dict_val)

# insts_list_test = get_keys_list(insts_dict_test)
# capts_list_test = get_keys_list(capts_dict_test)

### Procesar datos de categorias

In [ ]:
def build_id_name_mappings(categories):
    """
    Build dictionaries to map between category_id and class name.

    Parameters:
        categories (list[dict]): Output of load_categories().

    Returns:
        tuple:
            - id_to_name: {category_id: class_name}
            - name_to_id: {class_name: category_id}
    """
    # Map category_id -> name
    id_to_name = {cat["id"]: cat["name"] for cat in categories}

    # Map class name -> category_id
    name_to_id = {cat["name"]: cat["id"] for cat in categories}

    return id_to_name, name_to_id

def build_label_index_mappings(categories):
    """
    Build mappings between COCO category_id and contiguous label indices
    for training (0 .. num_classes-1).

    Parameters:
        categories (list[dict]): Output of load_categories().

    Returns:
        tuple:
            - id_to_idx: {category_id: contiguous_index}
            - idx_to_id: {contiguous_index: category_id}
            - classes:  sorted list of all category_ids
    """
    # Collect all category_ids and sort for stable indexing
    classes = sorted(cat["id"] for cat in categories)

    # Map category_id -> contiguous index
    id_to_idx = {cid: idx for idx, cid in enumerate(classes)}

    # Map contiguous index -> original category_id
    idx_to_id = {idx: cid for idx, cid in enumerate(classes)}

    return id_to_idx, idx_to_id, classes

def parse_index_mappings(json_path):
    # Load the category list from the COCO annotation JSON
    categories = load_categories(json_path)

    # Build mappings: id<->name
    id_to_name, name_to_id = build_id_name_mappings(categories)

    # Build mappings: id<->index and the sorted class list
    id_to_idx, idx_to_id, classes = build_label_index_mappings(categories)

    # Return all necessary mappings for category analysis
    return id_to_name, name_to_id, id_to_idx, idx_to_id, classes

In [ ]:
# id_to_name, name_to_id, id_to_idx, idx_to_id, classes = parse_index_mappings(path_instances_train)

# print(f"Num classes: {len(classes)}")
# print(f"Example: 17 -> idx {id_to_idx[17]} -> name {id_to_name[17]}")
# print(classes)

### Obtener lista de archivos de imágenes

In [ ]:
# Routes to images folders

path_images_train = f"{ROOT}\\raw_data\\images\\train2017\\"
path_images_val = f"{ROOT}\\raw_data\\images\\val2017\\"
path_images_test = f"{ROOT}\\raw_data\\images\\test2017\\"

In [ ]:
# Function to get a sorted list of images files in a given route

def get_images_list(images_route):
    images_list = os.listdir(images_route) # List of all images in the route
    images_list.sort()
    
    # print(len(images_list)) # Print length of the list
    
    # for i in range(len(images_list)):
      #   print(images_list[i])

    return images_list

images_list_train = get_images_list(path_images_train)
# images_list_val = get_images_list(path_images_val)
# images_list_test = get_images_list(path_images_test)

### Alineación entre de datos entre los diccionarios y las imágenes y agregar dirección completa de cada imagen

In [ ]:
# Function to align two dictionaries and a list by their common keys

def align_two_dicts_and_list(dict_insts, list_insts, dict_capts, list_capts, images_list, path_images):

    new_list = set(list_insts) & set(list_capts) & set(images_list)

    del_count_1 = 0
    del_count_2 = 0
    del_count_3 = 0
    
    # Deleting extra keys in both dictionaries
    for key1 in list_insts.copy():
        if key1 not in new_list:
            del_count_1 += 1
            list_insts.remove(key1)
            del dict_insts[key1]
    
    for key2 in list_capts.copy():
        if key2 not in new_list:
            del_count_2 += 1
            list_capts.remove(key2)
            del dict_capts[key2]

    for image in images_list.copy():
        if image not in new_list:
            del_count_3 += 1
            images_list.remove(image)
            # if os.path.isfile(os.path.join(path_images, image)):
                # os.remove(os.path.join(path_images_files, image_file))
    
    print(f"Deleted from Instances Dictionary: {del_count_1}")
    print(f"Deleted from Captions Dictionary: {del_count_2}")
    print(f"Deleted from Images List: {del_count_3}")

    return dict_insts, list_insts, dict_capts, list_capts, images_list

In [ ]:
insts_dict_train, insts_list_train, capts_dict_train, capts_list_train, images_list_train = align_two_dicts_and_list(insts_dict_train, insts_list_train, capts_dict_train, capts_list_train, images_list_train, path_images_train)

In [ ]:
# Be sure that all three have the same keys after alignment

print(len(insts_list_train))
print(len(capts_list_train))
print(len(images_list_train))
print(set(insts_list_train) == set(capts_list_train) == set(images_list_train))

# Print aligned contents

# for i in range(len(insts_list_train)):
  #   print(f"{insts_list_train[i]} | {capts_list_train[i]} | {images_list_train[i]}")

In [ ]:
# Function to add path to each image in the images list

def add_path_to_images_list(images_list, path_images):
    new_list = []

    for i in range(len(images_list)):
        new_list.append(path_images + images_list[i])
    #     print(new_list[i])

    images_list = new_list
    return images_list

In [ ]:
images_list_train = add_path_to_images_list(images_list_train, path_images_train)
# images_list_val = add_path_to_images_list(images_list_val, path_images_val)
# images_list_test = add_path_to_images_list(images_list_test, path_images_test)

In [ ]:
print(len(images_list_train))
# print(len(images_list_val))
# print(len(images_list_test))

In [ ]:
# Ploting a image from the training set

image_path = images_list_train[10]
image = cv2.imread(image_path)
print(image.shape)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
plt.imshow(image)

## Implemetación de red convolucional

### Dataset

In [ ]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    # transforms.RandomCrop((128, 128)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.RandomRotation(10),
    # transforms.ColorJitter(brightness = 0.1, contrast = 0.1, saturation = 0.1, hue = 0.1),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

In [ ]:
class RoboCapDataset(Dataset):
    def __init__(self, image_paths, transform = None):
        self.image_paths = image_paths
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image

In [ ]:
train_dataset = RoboCapDataset(images_list_train, transform = transform)
# val_dataset = RoboCapDataset(images_list_val, transform = transform)
# test_dataset = RoboCapDataset(images_list_test, transform = transform)

### Dataloaders

In [ ]:
train_dataloader = DataLoader(train_dataset, batch_size = 25, shuffle = False)
# val_dataloader = DataLoader(val_dataset, batch_size = 25, shuffle = False)
# test_dataloader = DataLoader(test_dataset, batch_size = 25, shuffle = False)

### Red convolucional

In [ ]:
# class RoboCapCVModel(L.LightningModule):

#     def __init__(self, backbone, learning_rate, num_classes):
#         super().__init__()

#         self.learning_rate = learning_rate

#         # Red preentrenada sin la capa fc
#         self.backbone = backbone

#         # Nuevo clasificador lineal -> devuelve logits
#         self.classifier = nn.Linear(backbone.fc.in_features if hasattr(backbone, "fc") else in_features,
#                                     num_classes)

#         self.train_acc = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
#         self.val_acc   = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)
#         self.test_acc  = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes)

#     def forward(self, x):
#         """
#         Forward: regresa logits (NO softmax)
#         """
#         features = self.backbone(x)      # vector de features
#         logits   = self.classifier(features)  # logits por clase
#         return logits

#     def _shared_step(self, batch):
#         features, true_labels = batch

#         logits = self(features)  # logits

#         loss = F.cross_entropy(logits, true_labels)
#         predicted_labels = torch.argmax(logits, dim=1)
#         return loss, true_labels, predicted_labels

#     def training_step(self, batch, batch_idx):
#         loss, true_labels, predicted_labels = self._shared_step(batch)
#         self.log("train_loss", loss)
#         self.train_acc(predicted_labels, true_labels)
#         self.log("train_acc", self.train_acc, prog_bar=True, on_epoch=True, on_step=False)
#         return loss

#     def validation_step(self, batch, batch_idx):
#         loss, true_labels, predicted_labels = self._shared_step(batch)
#         self.log("val_loss", loss, prog_bar=True)
#         self.val_acc(predicted_labels, true_labels)
#         self.log("val_acc", self.val_acc, prog_bar=True)

#     def test_step(self, batch, batch_idx):
#         with torch.no_grad():
#             loss, true_labels, predicted_labels = self._shared_step(batch)
#             self.test_acc(predicted_labels, true_labels)
#             self.log("test_acc", self.test_acc)

#     def configure_optimizers(self):
#         optimizer = torch.optim.RMSprop(self.parameters(), lr=self.learning_rate)
#         return optimizer

In [ ]:
class RoboCapFeatureExtractor(L.LightningModule):

    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone

        # Como solo queremos usarlo para inferencia (no entrenar), congelamos pesos
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, x):
        """
        x: batch de imágenes (B, C, H, W)
        return: logits/embeddings (B, D)
        """
        features = self.backbone(x)  # vector de tamaño in_features
        return features

    # Opcional: para usar Trainer.predict
    def predict_step(self, batch, batch_idx):
        images = batch  # recuerda: tu dataloader ya no regresa labels
        with torch.no_grad():
            feats = self(images)
        return feats


In [ ]:
# resnet101_model = torch.hub.load("pytorch/vision", "resnet101", weights = None)
resnet101_model = models.resnet101(weights = models.ResNet101_Weights.IMAGENET1K_V1)

# Reemplazar la última capa
in_features = resnet101_model.fc.in_features

# Convertir la capa fc (Clasificador) en identidad (extraer logits)
resnet101_model.fc = nn.Identity()

# Sin Fine-Tuning (congelar todas las capas excepto la fc)
for name, param in resnet101_model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

In [ ]:
# Crear objeto de modelo
feature_extractor = RoboCapFeatureExtractor(backbone = resnet101_model)

In [ ]:
trainer = L.Trainer(accelerator="cpu", devices=1, logger=False)

# Por ejemplo, sacamos embeddings del conjunto de entrenamiento
features_list = trainer.predict(model=feature_extractor,
                                dataloaders=train_dataloader)

# features_list es una lista de tensores (uno por batch)
all_features = torch.cat(features_list, dim=0)  # shape: (N_imágenes, in_features)

In [ ]:
all_features = []

feature_extractor.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
feature_extractor.to(device)

with torch.no_grad():
    for images in train_dataloader:
        images = images.to(device)
        feats = feature_extractor(images)  # (B, in_features)
        all_features.append(feats.cpu())

all_features = torch.cat(all_features, dim=0)
